importing the neccessary libraries

In [ ]:
import pandas as pd
import re
import numpy as np
from scipy.stats import chi2_contingency
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests

connecting the colab to the drive for secure access of the data, then making the df with it.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
sales = pd.read_csv('/content/drive/MyDrive/DA_projects/Maruti_Suzuki_Sales.csv')
product = pd.read_csv('/content/drive/MyDrive/DA_projects/Maruti_Suzuki_Products.csv')
dealers = pd.read_csv('/content/drive/MyDrive/DA_projects/Maruti_Suzuki_Dealers.csv')

In [ ]:
sales.head(20)
#product.head()
#dealers.head()

,SaleDate,InvoiceID,ProductID,FuelType,Transmission,Color,DealerCode,Channel,UnitsSold,ExShowroomPrice,DiscountApplied,OnRoadPrice,PaymentMode,FinancePartner,CustomerType,SatisfactionScore,BookingToDeliveryDays
0,18-05-2023,GU-AMR-20230518-00001,MSZ-CIAZ-2014-001,Petrol,AMT,Beige,AMR,Arena,1,932877,15000,959148,Loan,HDFC,Individual,84.0,12.0
1,21-03-2024,KA-PRT-20240321-00001,MSZ-IGNI-2016-003,Petrol,AMT,Beige,PRT,NEXA,1,649790,5000,673433,Loan,HDFC,Fleet,90.0,7.0
2,02-10-2022,KA-PRT-20221002-00001,MSZ-JIMN-2019-003,Petrol,Automatic,Blue,PRT,NEXA,1,1678291,20000,1675155,Lease,NaN,Individual,73.0,51.0
3,18-05-2023,HA-VPL-20230518-00001,MSZ-FRON-2023-001,Petrol,CVT,Black,VPL,Arena,1,839367,20000,872974,Loan,IDFC,Individual,79.0,30.0
4,29-05-2021,TA-AMB-20210529-00001,MSZ-FR0N-2023-001,Petrol,Automatic,Black,AMB,NEXA,1,â‚¹871980,20000,895414,Lease,NaN,Individual,84.0,10.0
5,18-06-2022,TA-AMB-20220618-00001,MSZ-INVI-2021-003,Petrol Hybrid,Automatic,Blue,AMB,NEXA,1,1422565,15000,1443229,Cash,NaN,Corporate,100.0,41.0
6,15-09-2021,GU-KAT-20210915-00001,MSZ-BALE-2015-002,CNG,AMT,Red,KAT,Arena,1,914740,10000,927457,Cash,NaN,Fleet,96.0,58.0
7,17-02-2023,HA-VPL-20230217-00001,MSZ-SWIF-2005-002,Petrol,Manual,Black,VPL,NEXA,1,â‚¹734660,20000,737622,Cash,NaN,Fleet,83.0,58.0
8,12-07-2023,HA-VPL-20230712-00001,MSZ-JIMN-2019-001,Petrol,Manual,Beige,VPL,NEXA,1,1364792,25000,1397800,Lease,NaN,Corporate,91.0,32.0
9,18-03-2022,DE-MAG-20220318-00001,MSZ-WAGO-1999-003,Petrol,CVT,Blue,MAG,Arena,1,571197,20000,590442,Lease,NaN,Corporate,86.0,20.0


To understand the dimension of the data

In [ ]:
print(f"The dimension of the sales data is {sales.shape}")
print(f"The dimension of the product data is {product.shape}")
print(f"The dimension of the dealers data is {dealers.shape}")

The dimension of the sales data is (800000, 17)
The dimension of the product data is (57, 6)
The dimension of the dealers data is (16, 4)


To understand about the data types and the nulls in the each indivdual columns

In [ ]:
sales.info()
product.info()
dealers.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 800000 entries, 0 to 799999
Data columns (total 17 columns):
 #   Column                 Non-Null Count   Dtype  
---  ------                 --------------   -----  
 0   SaleDate               800000 non-null  object 
 1   InvoiceID              799255 non-null  object 
 2   ProductID              798430 non-null  object 
 3   FuelType               800000 non-null  object 
 4   Transmission           800000 non-null  object 
 5   Color                  800000 non-null  object 
 6   DealerCode             800000 non-null  object 
 7   Channel                800000 non-null  object 
 8   UnitsSold              800000 non-null  int64  
 9   ExShowroomPrice        800000 non-null  object 
 10  DiscountApplied        800000 non-null  int64  
 11  OnRoadPrice            800000 non-null  int64  
 12  PaymentMode            800000 non-null  object 
 13  FinancePartner         228277 non-null  object 
 14  CustomerType           800000 non-nu

## NULLS IN THE DATA
### Sales  
- InvoiceId, ProductID has missing values (null values) which are concerning issue. since ProductID is used to map to the characteristics of the product from the product table and the invoice ID needs to be unique which needs to further verified
- Finance Partner has only 228277 datapoints which only covers the 28.5% of the data
- 2380 customerSatisfactionScore and 1530 BookingToDeliveryDays are missing which further needs to be investigated is it missing at random or the data quality issue

### Products
- No missing value is found in any of the column

### Dealers
- No missing value is found in any of the column

## Data- types analysis:
### Sales
- ExShowRoomPrice instead of being float or int it is object, so the issue needs to be sorted.
- The satisfactoryScore and BookingToDeliveryDays are in float format they needs to be int format. Then it means that they should be some NaN values that causes it. This is verified by the null values present in the data.

### Products
- All the columns are in the expected business meaning.

### Dealers
- All the columns are in the expected business meaning




In [ ]:
sales["SaleDate"].unique()

array(['18-05-2023', '21-03-2024', '02-10-2022', ..., '04/21/2023',
       '08/31/2023', '06/19/2024'], dtype=object)

When lookig into the unique values of the SaleDate we could see that the format of the SaleDate is different and needs to be formatted under an uniform format.

## DATA CLEANING AND HANDLING NULLS

- First looking into ExShowRoomPrice, when we try to convert that int, it gives error because the data has other unwanted data included with the numerical data.
like â‚¹ or some other patterns like INR

In [ ]:
price = pd.to_numeric(sales['ExShowroomPrice'],errors='coerce')
bad_rows = sales[price.isna() & sales['ExShowroomPrice'].notna()]
print(bad_rows['ExShowroomPrice'].value_counts())

ExShowroomPrice
â‚¹1501637    2
â‚¹1174694    2
â‚¹1381612    2
â‚¹861119     2
â‚¹455819     2
             ..
â‚¹1374045    1
â‚¹1140566    1
â‚¹1012737    1
â‚¹813700     1
472421 INR    1
Name: count, Length: 19695, dtype: int64


- so a function is defined where the cleaning will be done in such a way that other than numbers all others will be removed. The showroom price would be returned in the int format.

In [ ]:
def clean_price(value):
  text = str(value)
  numbers_only = re.sub(r'[^0-9]','',text)
  if numbers_only == '':
    return float('nan')
  return int(numbers_only)

sales['ExShowroomPrice'] = sales['ExShowroomPrice'].apply(clean_price)
sales['ExShowroomPrice'] = pd.to_numeric(sales['ExShowroomPrice'])
print(sales['ExShowroomPrice'].head())
print(sales['ExShowroomPrice'].dtype)

0     932877
1     649790
2    1678291
3     839367
4     871980
Name: ExShowroomPrice, dtype: int64
int64


- Making the SaleDate in ISO 8601 stanadard before moving into further analysis YYYY-MM-DD

In [ ]:
sales['SaleDate'] = pd.to_datetime(sales['SaleDate'],format='mixed',errors= 'coerce')

print(sales['SaleDate'].unique())
print(sales['SaleDate'].isna().sum())
print(sales['SaleDate'].shape)

<DatetimeArray>
['2023-05-18 00:00:00', '2024-03-21 00:00:00', '2022-02-10 00:00:00',
 '2021-05-29 00:00:00', '2022-06-18 00:00:00', '2021-09-15 00:00:00',
 '2023-02-17 00:00:00', '2023-12-07 00:00:00', '2022-03-18 00:00:00',
 '2021-10-09 00:00:00',
 ...
 '2020-09-01 00:00:00', '2020-06-02 00:00:00', '2025-01-11 00:00:00',
 '2025-03-10 00:00:00', '2020-10-02 00:00:00', '2025-02-11 00:00:00',
 '2025-01-05 00:00:00', '2025-03-06 00:00:00', '2025-01-06 00:00:00',
 '2025-01-07 00:00:00']
Length: 1880, dtype: datetime64[ns]
0
(800000,)


- satisfaction score 2380 missing, evaluation has to be done to see if it is missing at random or data quality issue
- Basic analysis is done here

In [ ]:
sat_deal = sales[sales['SatisfactionScore'].isna()].groupby('DealerCode').size()
sat_cha = sales[sales['SatisfactionScore'].isna()].groupby('Channel').size()
sat_cus = sales[sales['SatisfactionScore'].isna()].groupby('CustomerType').size()
sat_uni = sales[sales['SatisfactionScore'].isna()].groupby('UnitsSold').size()
deal = sales.groupby('DealerCode').size()
cha = sales.groupby('Channel').size()
cus = sales.groupby('CustomerType').size()
uni = sales.groupby('UnitsSold').size()
# normalize and see the contribution of the missing value
print(sat_deal/deal)
print(sat_cha/cha)
print(sat_cus/cus)
print(sat_uni/uni)

DealerCode
AMB    0.002589
AMR    0.003242
CMP    0.002835
DDM    0.002989
IND    0.002961
KAT    0.002866
KHV    0.003227
KTL    0.002713
MAG    0.002749
PRM    0.002954
PRT    0.003126
RAN    0.003247
SAG    0.002900
TRS    0.003174
VAR    0.002912
VPL    0.003116
dtype: float64
Channel
Arena    0.00290
NEXA     0.00305
dtype: float64
CustomerType
Corporate     0.003007
Fleet         0.002842
Individual    0.003077
dtype: float64
UnitsSold
1    0.002992
2    0.002586
3    0.003765
dtype: float64


Further perorming statistical analysis on the missing data to come to conlusion if it is MCAR (missing completely at random)

In [ ]:
sales["Year"] = sales["SaleDate"].dt.year
sales["Month"] = sales["SaleDate"].dt.month
sales["SatisfactionMissing"] = sales["SatisfactionScore"].isna()
sales.groupby("Year")["SatisfactionMissing"].mean() * 100

,SatisfactionMissing
Year,
2020,0.262576
2021,0.291372
2022,0.286592
2023,0.324202
2024,0.305072
2025,0.334533


### PERFORMING STATISTICAL ANALYSIS ON MISSING DATA

- First finding which columns have missing values
- Then trying to bin the column based on the data type

In [ ]:
missing_summary = (sales.isna().sum().sort_values(ascending=False))
missing_summary = missing_summary[missing_summary > 0]
print(missing_summary)

FinancePartner           571723
SatisfactionScore          2380
ProductID                  1570
BookingToDeliveryDays      1530
InvoiceID                   745
dtype: int64


In [ ]:
categorical_cols = ["DealerCode","Channel","CustomerType","PaymentMode","FinancePartner","FuelType","Transmission","Color"]
numerical_cols = ["UnitsSold","ExShowroomPrice","DiscountApplied","OnRoadPrice","BookingToDeliveryDays","SatisfactionScore"]
date_col = "SaleDate"

- For categorical variables we used Chi squared to check the significance of missing within the given missing variable and corresponding other variable in the sales dataframe
- If p<0.05 then the data is not missing in random
- Otherwide the data is missing in completely random

In [ ]:
def categorical_missing_tests(df, missing_col, categorical_cols):
    results = []
    missing_flag = f"{missing_col}_Missing"
    df = df.copy()
    df[missing_flag] = df[missing_col].isna()
    for col in categorical_cols:
        # preventing the variable self testing
        if col == missing_col:
            continue
        # Remove missing value in the cross variable
        temp = df[[missing_flag, col]].dropna()
        # Need at least 2 categories
        if temp[col].nunique() < 2:
            continue
        table = pd.crosstab(temp[missing_flag],temp[col])
        try:
            chi2, p, dof, expected = chi2_contingency(table)
            results.append({
                "MissingVariable": missing_col,
                "TestVariable": col,
                "Test": "Chi-square",
                "Statistic": chi2,
                "p_value": p,
                "Significant": p <= 0.05
            })
        except Exception as e:
            results.append({
                "MissingVariable": missing_col,
                "TestVariable": col,
                "Test": "Chi-square",
                "Statistic": np.nan,
                "p_value": np.nan,
                "Significant": np.nan
            })
    return pd.DataFrame(results)

- Similarly for missing numerical variables like CustomerStatisfactionScore we use Mann-whitney U test for it

In [ ]:
def numerical_missing_tests(df, missing_col, numerical_cols):
    results = []
    missing_flag = f"{missing_col}_Missing"
    df = df.copy()
    df[missing_flag] = df[missing_col].isna()
    for col in numerical_cols:
        if col == missing_col:
            continue
        temp = df[[missing_flag, col]].dropna()
        missing_group = temp.loc[
            temp[missing_flag] == True,
            col
        ]
        observed_group = temp.loc[
            temp[missing_flag] == False,
            col
        ]
        # Need both groups
        if len(missing_group) < 2 or len(observed_group) < 2:
            continue
        try:
            statistic, p = mannwhitneyu(
                missing_group,
                observed_group,
                alternative="two-sided"
            )
            results.append({
                "MissingVariable": missing_col,
                "TestVariable": col,
                "Test": "Mann-Whitney U",
                "Statistic": statistic,
                "p_value": p,
                "Significant": p <= 0.05
            })
        except Exception:
            results.append({
                "MissingVariable": missing_col,
                "TestVariable": col,
                "Test": "Mann-Whitney U",
                "Statistic": np.nan,
                "p_value": np.nan,
                "Significant": np.nan
            })
    return pd.DataFrame(results)

- Further analysis is done over the range of year and months also

In [ ]:
def temporal_missing_analysis(df, missing_col, date_col):
    temp = df.copy()
    temp[date_col] = pd.to_datetime(
        temp[date_col],
        errors="coerce"
    )
    temp["Year"] = temp[date_col].dt.year
    yearly = (
        temp.groupby("Year")[missing_col]
        .apply(lambda x: x.isna().mean() * 100)
        .reset_index(name="MissingRate_%")
    )
    temp["YearMonth"] = temp[date_col].dt.to_period("M")
    monthly = (
        temp.groupby("YearMonth")[missing_col]
        .apply(lambda x: x.isna().mean() * 100)
        .reset_index(name="MissingRate_%")
    )
    return yearly, monthly

- Since multiple statistical tests were performed, the Benjamini–Hochberg (BH) procedure was applied to control the false discovery rate (FDR). The adjusted p-values were used to identify statistically significant associations while reducing the risk of false-positive results

In [ ]:
def missing_summary(df):
    results = []
    total_rows = len(df)
    for col in df.columns:
        missing_count = df[col].isna().sum()
        if missing_count > 0:
            results.append({
                "Variable": col,
                "TotalRecords": total_rows,
                "MissingRecords": missing_count,
                "MissingRate_%": (
                    missing_count / total_rows
                ) * 100
            })
    return pd.DataFrame(results).sort_values(
        "MissingRate_%",
        ascending=False
    )
summary = missing_summary(sales)
print(summary)

                Variable  TotalRecords  MissingRecords  MissingRate_%
2         FinancePartner        800000          571723      71.465375
3      SatisfactionScore        800000            2380       0.297500
1              ProductID        800000            1570       0.196250
4  BookingToDeliveryDays        800000            1530       0.191250
0              InvoiceID        800000             745       0.093125


In [ ]:
all_results = []
missing_variables = sales.columns[
    sales.isna().any()
].tolist()
for missing_col in missing_variables:
    print(f"\nAnalyzing: {missing_col}")
    # Categorical tests
    cat_results = categorical_missing_tests(
        sales,
        missing_col,
        categorical_cols
    )
    # Numerical tests
    num_results = numerical_missing_tests(
        sales,
        missing_col,
        numerical_cols
    )
    # Combine
    results = pd.concat(
        [cat_results, num_results],
        ignore_index=True
    )
    all_results.append(results)
all_tests = pd.concat(
    all_results,
    ignore_index=True)
mask = all_tests["p_value"].notna()

all_tests.loc[mask, "p_adjusted"] = multipletests(
    all_tests.loc[mask, "p_value"],
    method="fdr_bh"
)[1]

all_tests["Significant_Adjusted"] = (
    all_tests["p_adjusted"] <= 0.05
)
print(all_tests)


Analyzing: InvoiceID

Analyzing: ProductID

Analyzing: FinancePartner

Analyzing: SatisfactionScore

Analyzing: BookingToDeliveryDays
          MissingVariable       TestVariable            Test     Statistic  \
0               InvoiceID         DealerCode      Chi-square  1.606768e+01   
1               InvoiceID            Channel      Chi-square  5.886250e-03   
2               InvoiceID       CustomerType      Chi-square  5.992283e-01   
3               InvoiceID        PaymentMode      Chi-square  1.638932e+00   
4               InvoiceID     FinancePartner      Chi-square  3.071702e+00   
..                    ...                ...             ...           ...   
62  BookingToDeliveryDays          UnitsSold  Mann-Whitney U  6.126880e+08   
63  BookingToDeliveryDays    ExShowroomPrice  Mann-Whitney U  6.188313e+08   
64  BookingToDeliveryDays    DiscountApplied  Mann-Whitney U  6.063558e+08   
65  BookingToDeliveryDays        OnRoadPrice  Mann-Whitney U  6.191095e+08   
66  Boo

In [ ]:
significant_results = all_tests[
    all_tests["p_value"] <= 0.05
].sort_values("p_value")
print(significant_results)

      MissingVariable     TestVariable            Test     Statistic  \
31     FinancePartner      PaymentMode      Chi-square  6.409376e+05   
36     FinancePartner  ExShowroomPrice  Mann-Whitney U  6.502175e+10   
38     FinancePartner      OnRoadPrice  Mann-Whitney U  6.502233e+10   
50  SatisfactionScore  ExShowroomPrice  Mann-Whitney U  9.228297e+08   
52  SatisfactionScore      OnRoadPrice  Mann-Whitney U  9.228579e+08   
20          ProductID     Transmission      Chi-square  7.955238e+00   

     p_value  Significant  p_adjusted  Significant_Adjusted  
31  0.000000         True    0.000000                  True  
36  0.012174         True    0.259289                 False  
38  0.012387         True    0.259289                 False  
50  0.019221         True    0.259289                 False  
52  0.019350         True    0.259289                 False  
20  0.046946         True    0.524230                 False  


In [ ]:
non_significant_results = all_tests[
    all_tests["p_value"] > 0.05
]
print(non_significant_results)

          MissingVariable       TestVariable            Test     Statistic  \
0               InvoiceID         DealerCode      Chi-square  1.606768e+01   
1               InvoiceID            Channel      Chi-square  5.886250e-03   
2               InvoiceID       CustomerType      Chi-square  5.992283e-01   
3               InvoiceID        PaymentMode      Chi-square  1.638932e+00   
4               InvoiceID     FinancePartner      Chi-square  3.071702e+00   
..                    ...                ...             ...           ...   
62  BookingToDeliveryDays          UnitsSold  Mann-Whitney U  6.126880e+08   
63  BookingToDeliveryDays    ExShowroomPrice  Mann-Whitney U  6.188313e+08   
64  BookingToDeliveryDays    DiscountApplied  Mann-Whitney U  6.063558e+08   
65  BookingToDeliveryDays        OnRoadPrice  Mann-Whitney U  6.191095e+08   
66  BookingToDeliveryDays  SatisfactionScore  Mann-Whitney U  5.946850e+08   

     p_value  Significant  p_adjusted  Significant_Adjusted  
0

- By using the significant test we have seen that the missing in FinancePartner are not at random and vary significantly across ExshowroomPrice, PaymentMode and OnRoadPrice.
- Similary for statisfactionScor it is not missing at random in ExShowroomPrice and OnRoadPrice
- In the case of ProductID the missing is not at random in Transmission of the vehicle

In [ ]:
sig = all_tests[
    all_tests["Significant_Adjusted"] == True
].copy()

sig[[
    "MissingVariable",
    "TestVariable",
    "Test",
    "p_value",
    "p_adjusted",
    "Significant_Adjusted"
]].sort_values("p_adjusted")

,MissingVariable,TestVariable,Test,p_value,p_adjusted,Significant_Adjusted
31,FinancePartner,PaymentMode,Chi-square,0.0,0.0,True


- Finally after significnt adjusted p value it is statsitically evident that FinancePartner is not missing at random in the payment mode which further needs investigation.
- All other missing values are statsitically significant to claim that they are missing completely at random

#### Investigation of PaymentMethod and FinancePartner

In [ ]:
print(sales[sales["FinancePartner"].isna()]["PaymentMode"].value_counts())
print(sales[sales["FinancePartner"].notna()]["PaymentMode"].value_counts())
print(sales["PaymentMode"].value_counts())

PaymentMode
Cash     267566
Lease    266348
Loan      37809
Name: count, dtype: int64
PaymentMode
Loan    228277
Name: count, dtype: int64
PaymentMode
Cash     267566
Lease    266348
Loan     266086
Name: count, dtype: int64


- It is evident that FinancePartner is only present when the PaymentMode is Loan and even in the case of the loan there is no financepartners for 37809 datapoints.
- ProductID is missing without it cannot be mapped to the product characteristics Product table. If it is null then we need to remove
- removing the null in the Product table

In [ ]:
sales = sales[sales['ProductID'].notna()]
sales.shape

(798430, 20)

- InvoiceID - each date can have multiple invoices but have unique id so it can be choosen as unique identifier for the given data
- The InvoiceID is in two different formats as well
- FORMAT 1
  - AB-CDE-00000000-00001
  - AB defines the two letter state code of the Dealer
  - CDE defines the three letter Dealer code
  - 00000000 defines the date in the format of YYYY-MM-DD
  - 00001 defines the number of the invoice made in a given day ordered in the asceding based on the index of the data.
- FORMAT 2
 - ABC/0000/00
 - where ABC is three letter dealer code
 - Rest all are randomly assigned numbers   

In [ ]:
sales[sales['InvoiceID'].str.startswith('GU-AMR-20230518', na=False)]
#sales['InvoiceID'].head(20)

,SaleDate,InvoiceID,ProductID,FuelType,Transmission,Color,DealerCode,Channel,UnitsSold,ExShowroomPrice,DiscountApplied,OnRoadPrice,PaymentMode,FinancePartner,CustomerType,SatisfactionScore,BookingToDeliveryDays,Year,Month,SatisfactionMissing
0,2023-05-18,GU-AMR-20230518-00001,MSZ-CIAZ-2014-001,Petrol,AMT,Beige,AMR,Arena,1,932877,15000,959148,Loan,HDFC,Individual,84.0,12.0,2023,5,False
4422,2023-05-18,GU-AMR-20230518-00002,MSZ-FR0N-2023-001,Petrol,Automatic,Black,AMR,Arena,1,900494,15000,943399,Cash,NaN,Fleet,91.0,48.0,2023,5,False
35886,2023-05-18,GU-AMR-20230518-00003,MSZ-ALTO-2000-002,Petrol,Automatic,Black,AMR,Arena,3,327660,0,352052,Loan,Yes Bank,Individual,93.0,6.0,2023,5,False
65237,2023-05-18,GU-AMR-20230518-00004,MSZ-GRAN-2022-002,Petrol Hybrid,CVT,Blue,AMR,Arena,1,1577618,10000,1607485,Lease,NaN,Fleet,88.0,25.0,2023,5,False
85546,2023-05-18,GU-AMR-20230518-00005,MSZ-GRAN-2022-003,Petrol Hybrid,Manual,Grey,AMR,NEXA,1,1509243,0,1547011,Lease,NaN,Individual,93.0,56.0,2023,5,False
86707,2023-05-18,GU-AMR-20230518-00006,MSZ-JIMN-2019-002,Petrol,AMT,Silver,AMR,NEXA,1,1556411,0,1595029,Loan,IDFC,Fleet,84.0,54.0,2023,5,False
91198,2023-05-18,GU-AMR-20230518-00007,MSZ-CIAZ-2014-001,Petrol,CVT,Beige,AMR,NEXA,1,916504,5000,930090,Lease,NaN,Individual,81.0,59.0,2023,5,False
131919,2023-05-18,GU-AMR-20230518-00008,MSZ-CIAZ-2014-003,Petrol,AMT,Silver,AMR,NEXA,1,1119833,25000,1150113,Cash,NaN,Corporate,82.0,58.0,2023,5,False
137807,2023-05-18,GU-AMR-20230518-00009,MSZ-BREZ-2016-002,Petrol,AMT,Grey,AMR,Arena,2,958522,10000,969476,Lease,NaN,Fleet,90.0,12.0,2023,5,False
139401,2023-05-18,GU-AMR-20230518-00010,MSZ-FRON-2023-002,Petrol,Automatic,Beige,AMR,NEXA,1,971124,25000,1008553,Cash,NaN,Individual,89.0,7.0,2023,5,False


Since only 745 data are missing, which is close to only 0.1 % of the entire dataset we can drop it instead of creating new invoice id.

In [ ]:
sales = sales[sales['InvoiceID'].notna()]
sales.shape

(797687, 20)

Finally after handling nulls we and data preprocessing we will be moving ahead with 797687 which is 99.71% of the data.  

Validating the relationship between the tables

In [ ]:
print("Products ProductID unique:",product["ProductID"].is_unique)
print("Duplicate ProductIDs:",product["ProductID"].duplicated().sum())
invalid_product = sales.loc[sales["ProductID"].notna() &~sales["ProductID"]
                            .isin(product["ProductID"])]
print("Unmatched ProductIDs in Sales:",len(invalid_product))

print("Dealers DealerCode unique:",dealers["DealerCode"].is_unique)

print("Duplicate DealerCodes:",dealers["DealerCode"].duplicated().sum())
invalid_dealer = sales.loc[sales["DealerCode"].notna() &
    ~sales["DealerCode"].isin(dealers["DealerCode"])]
print("Unmatched DealerCodes in Sales:",
      len(invalid_dealer))

Products ProductID unique: True
Duplicate ProductIDs: 0
Unmatched ProductIDs in Sales: 0
Dealers DealerCode unique: True
Duplicate DealerCodes: 0
Unmatched DealerCodes in Sales: 0


Validating the UnitsSold column for unusual values

In [ ]:
sales["UnitsSold"].describe()

,UnitsSold
count,797687.000000
mean,1.119587
std,0.380895
min,1.000000
25%,1.000000
50%,1.000000
75%,1.000000
max,3.000000


In [ ]:
sales["UnitsSold"].value_counts()

,count
UnitsSold,
1,718166
2,63649
3,15872


In [ ]:
sales_with_products = sales.merge(
    product[["ProductID", "BasePriceVariant"]],
    on="ProductID",
    how="left"
)

sales_with_products["Base_vs_ExShowroom_Diff"] = (
    sales_with_products["BasePriceVariant"]
    - sales_with_products["ExShowroomPrice"]
)

sales_with_products[
    ["ProductID", "BasePriceVariant", "ExShowroomPrice",
     "DiscountApplied", "OnRoadPrice","Base_vs_ExShowroom_Diff"]
].head(20)

,ProductID,BasePriceVariant,ExShowroomPrice,DiscountApplied,OnRoadPrice,Base_vs_ExShowroom_Diff
0,MSZ-CIAZ-2014-001,950000,932877,15000,959148,17123
1,MSZ-IGNI-2016-003,648000,649790,5000,673433,-1790
2,MSZ-JIMN-2019-003,1620000,1678291,20000,1675155,-58291
3,MSZ-FRON-2023-001,902500,839367,20000,872974,63133
4,MSZ-FR0N-2023-001,902500,871980,20000,895414,30520
5,MSZ-INVI-2021-003,1512000,1422565,15000,1443229,89435
6,MSZ-BALE-2015-002,900000,914740,10000,927457,-14740
7,MSZ-SWIF-2005-002,700000,734660,20000,737622,-34660
8,MSZ-JIMN-2019-001,1425000,1364792,25000,1397800,60208
9,MSZ-WAGO-1999-003,540000,571197,20000,590442,-31197


In [ ]:
sales_with_products["Base_vs_ExShowroom_Diff"].describe()

,Base_vs_ExShowroom_Diff
count,797687.000000
mean,-5.571041
std,38535.614163
min,-113390.000000
25%,-25590.000000
50%,-57.000000
75%,25532.000000
max,113372.000000


In [ ]:
sales_with_products[
    sales_with_products["ExShowroomPrice"] <
    sales_with_products["BasePriceVariant"]
][
    ["ProductID", "BasePriceVariant", "ExShowroomPrice",
     "DiscountApplied", "OnRoadPrice"]
].shape

(398461, 5)

If discount applied on base variant since 398461 datapoints accord to the logic

In [ ]:
sales_with_products["Expected_ExShowroom_After_Discount"] = (
    sales_with_products["BasePriceVariant"]
    - sales_with_products["DiscountApplied"]
)

In [ ]:
sales_with_products["Price_Difference"] = (
    sales_with_products["ExShowroomPrice"]
    - sales_with_products["Expected_ExShowroom_After_Discount"]
)

In [ ]:
sales_with_products[
    [
        "BasePriceVariant",
        "DiscountApplied",
        "ExShowroomPrice",
        "Expected_ExShowroom_After_Discount",
        "Price_Difference"
    ]
].head(20)
print((sales_with_products["Price_Difference"]==0).sum())

8


Only in 8 instances in the entire dataset the when discount subtracted from baseprice gives ex showroom so the coorelation cannot be used further

In [ ]:
zero_discount = sales_with_products[
    sales_with_products["DiscountApplied"] == 0
]

zero_discount["Base_vs_ExShowroom_Diff"].describe()

,Base_vs_ExShowroom_Diff
count,133184.000000
mean,69.574198
std,38493.708641
min,-113374.000000
25%,-25434.000000
50%,135.000000
75%,25655.500000
max,113346.000000


In [ ]:
zero_discount[
    zero_discount["ExShowroomPrice"] < zero_discount["BasePriceVariant"]
][
    [
        "ProductID",
        "BasePriceVariant",
        "ExShowroomPrice",
        "DiscountApplied",
        "OnRoadPrice"
    ]
].shape

(66805, 5)

In 66805 datapoints the Exshowroom price is less than base price variant even when the discount is zero

In [ ]:
sales_with_products["Year"] = sales_with_products["SaleDate"].dt.year

sales_with_products.groupby("Year").agg(
    avg_base_price=("BasePriceVariant", "mean"),
    avg_exshowroom=("ExShowroomPrice", "mean"),
    avg_discount=("DiscountApplied", "mean"),
    avg_onroad=("OnRoadPrice", "mean")
)


,avg_base_price,avg_exshowroom,avg_discount,avg_onroad
Year,,,,
2020,882000.573256,882102.042487,12488.098700,909590.004038
2021,880595.378592,880599.730634,12496.119501,908051.941978
2022,883785.340429,883736.954791,12521.928314,911253.465869
2023,883599.583838,883572.495559,12489.815646,911056.482490
2024,883246.830651,883301.090093,12541.016945,910724.741077
2025,881713.316219,881583.958214,12502.224030,909017.427115


The average of discount given is same through out the years so that doesnt have any temporal changes in price

In [ ]:
sales["OnRoad_Premium"] = (
    sales["OnRoadPrice"] - sales["ExShowroomPrice"]
)

sales["OnRoad_Premium"].describe()

,OnRoad_Premium
count,797687.000000
mean,27470.180724
std,16758.398432
min,-10000.000000
25%,14986.000000
50%,27488.000000
75%,39945.000000
max,65000.000000


In [ ]:
sales.groupby("FuelType")["OnRoad_Premium"].mean()

,OnRoad_Premium
FuelType,
CNG,27500.546622
Petrol,27451.492796
Petrol Hybrid,27505.816206


In [ ]:
sales.groupby("FuelType")["OnRoad_Premium"].mean()

,OnRoad_Premium
FuelType,
CNG,27500.546622
Petrol,27451.492796
Petrol Hybrid,27505.816206


To check if the price mentioned here is for one quantity or the final invoice amount

In [ ]:
sales[
    ["InvoiceID", "UnitsSold", "ExShowroomPrice",
     "DiscountApplied", "OnRoadPrice"]
].sort_values("UnitsSold", ascending=False).head(20)

,InvoiceID,UnitsSold,ExShowroomPrice,DiscountApplied,OnRoadPrice
600403,KE-IND-20201122-00020,3,626594,25000,650628
50621,DE-RAN-20240207-00005,3,744768,20000,755063
688316,UT-PRM-20230614-00026,3,559047,0,607665
237200,UT-KTL-20230613-00004,3,335139,10000,347875
688306,UT-PRM-20210910-00021,3,1215655,20000,1237370
600409,KE-IND-20241013-00017,3,1522079,5000,1540935
167193,UT-KTL-20220404-00009,3,366095,10000,382315
167224,KA-SAG-20231116-00008,3,354134,0,400490
688244,KA-SAG-20220223-00027,3,903302,10000,943256
526012,DE-DDM-20220318-00015,3,620374,15000,666925


In [ ]:
price_variation = (
    sales.groupby("ProductID")["ExShowroomPrice"]
    .agg(["count", "nunique", "min", "max", "mean"])
)

price_variation.sort_values("nunique", ascending=False).head(20)

,count,nunique,min,max,mean
ProductID,,,,,
MSZ-GRAN-2022-001,14302,13774,1325252,1524748,1.424699e+06
MSZ-JIMN-2019-002,14190,13732,1395011,1604984,1.499986e+06
MSZ-INVI-2021-003,14182,13698,1406163,1617810,1.511184e+06
MSZ-JIMN-2019-001,14076,13601,1325260,1524737,1.425473e+06
MSZ-GRAN-2022-003,13992,13574,1506633,1733390,1.619592e+06
MSZ-INVI-2021-002,14080,13568,1302017,1497995,1.399592e+06
MSZ-GRAN-2022-002,13999,13546,1395024,1604992,1.500258e+06
MSZ-XL6-2019-001,14091,13498,1060200,1219786,1.140303e+06
MSZ-JIMN-2019-003,13892,13478,1506628,1733389,1.619670e+06


In [ ]:
sales["Year"] = sales["SaleDate"].dt.year

sales.groupby("Year")["ExShowroomPrice"].agg(
    ["count", "mean", "min", "max"]
)

,count,mean,min,max
Year,,,,
2020,120365,882102.042487,309251,1733390
2021,159129,880599.730634,309229,1733389
2022,159725,883736.954791,309235,1733358
2023,159313,883572.495559,309228,1733319
2024,159812,883301.090093,309231,1733362
2025,39343,881583.958214,309347,1733304


In [ ]:
sales.groupby("Transmission")["ExShowroomPrice"].agg(
    ["count", "mean", "min", "max"]
)

,count,mean,min,max
Transmission,,,,
AMT,198992,882910.827460,309228,1733358
Automatic,199832,882741.425843,309231,1733390
CVT,199161,882822.202992,309229,1733374
Manual,199702,882079.196453,309232,1733316


In [ ]:
sales.groupby("FuelType")["ExShowroomPrice"].agg(
    ["count", "mean", "min", "max"]
)

,count,mean,min,max
FuelType,,,,
CNG,210319,6.707607e+05,309232,1386719
Petrol,502871,8.735451e+05,309228,1733389
Petrol Hybrid,84497,1.464131e+06,1236922,1733390


In [ ]:
sales.groupby("Channel")["ExShowroomPrice"].agg(
    ["count", "mean", "min", "max"]
)

,count,mean,min,max
Channel,,,,
Arena,398904,882591.659703,309228,1733390
NEXA,398783,882684.480093,309229,1733389


In [ ]:
product.groupby("Model")["VariantCode"].nunique()

,VariantCode
Model,
Alto,3
Baleno,3
Baleno RS,3
Brezza,3
Celerio,3
Ciaz,3
Dzire,3
Eeco,3
Ertiga,3


In [ ]:
product[[
    "ProductID",
    "Model",
    "VariantCode",
    "BasePriceVariant"
]].sort_values("Model").head(30)

,ProductID,Model,VariantCode,BasePriceVariant
0,MSZ-ALTO-2000-001,Alto,V1,332500
1,MSZ-ALTO-2000-002,Alto,V2,350000
2,MSZ-ALTO-2000-003,Alto,V3,378000
20,MSZ-BALE-2015-003,Baleno,V3,972000
19,MSZ-BALE-2015-002,Baleno,V2,900000
18,MSZ-BALE-2015-001,Baleno,V1,855000
53,MSZ-BALE-2020-003,Baleno RS,V3,1188000
52,MSZ-BALE-2020-002,Baleno RS,V2,1100000
51,MSZ-BALE-2020-001,Baleno RS,V1,1045000
32,MSZ-BREZ-2016-003,Brezza,V3,1026000


In [ ]:
sales[
    sales["UnitsSold"] > 1
][
    [
        "InvoiceID",
        "UnitsSold",
        "ExShowroomPrice",
        "DiscountApplied",
        "OnRoadPrice"
    ]
].head(20)

,InvoiceID,UnitsSold,ExShowroomPrice,DiscountApplied,OnRoadPrice
22,TA-AMB-20221118-00001,2,1183775,5000,1220751
31,GU-AMR-20230918-00001,2,1488806,20000,1524508
35,GU-AMR-20201214-00001,2,745693,10000,782623
37,UT-PRM-20210418-00001,2,901337,25000,940417
41,TA-AMB-20200727-00001,2,1239136,5000,1295132
54,TA-KHV-20250131-00001,2,482784,25000,473447
64,UT-PRM-20210423-00001,2,348617,15000,375664
91,GU-KAT-20231012-00001,2,1181054,0,1212771
95,HA-VPL-20240428-00001,2,502247,0,531724
100,GU-KAT-20210518-00001,2,432677,5000,446236


In [ ]:
sales["SalesValue"] = sales["ExShowroomPrice"] * sales["UnitsSold"]

sales[["InvoiceID", "UnitsSold", "ExShowroomPrice", "SalesValue"]].head(20)

,InvoiceID,UnitsSold,ExShowroomPrice,SalesValue
0,GU-AMR-20230518-00001,1,932877,932877
1,KA-PRT-20240321-00001,1,649790,649790
2,KA-PRT-20221002-00001,1,1678291,1678291
3,HA-VPL-20230518-00001,1,839367,839367
4,TA-AMB-20210529-00001,1,871980,871980
5,TA-AMB-20220618-00001,1,1422565,1422565
6,GU-KAT-20210915-00001,1,914740,914740
7,HA-VPL-20230217-00001,1,734660,734660
8,HA-VPL-20230712-00001,1,1364792,1364792
9,DE-MAG-20220318-00001,1,571197,571197


To see if the given data has sales records between teh financial years of 2020-2025


In [ ]:
sales.groupby(
    [sales["SaleDate"].dt.year, sales["SaleDate"].dt.month]
).size()

SaleDate  SaleDate
2020      1            3815
          2            3823
          3            3720
          4           11929
          5           12246
                      ...  
2025      8            1272
          9            1278
          10           1275
          11           1274
          12           1321
Length: 72, dtype: int64

In [ ]:
sales.groupby(sales["SaleDate"].dt.year)["SaleDate"].agg(
    first_date="min",
    last_date="max",
    records="count"
)

,first_date,last_date,records
SaleDate,,,
2020,2020-01-04,2020-12-31,120365
2021,2021-01-01,2021-12-31,159129
2022,2022-01-01,2022-12-31,159725
2023,2023-01-01,2023-12-31,159313
2024,2024-01-01,2024-12-31,159812
2025,2025-01-01,2025-12-03,39343


In [ ]:
sales.groupby(
    [sales["SaleDate"].dt.year, sales["SaleDate"].dt.month]
).size().unstack(fill_value=0)

SaleDate,1,2,3,4,5,6,7,8,9,10,11,12
SaleDate,,,,,,,,,,,,
2020,3815,3823,3720,11929,12246,12014,12225,12224,12028,12244,11945,12152
2021,13592,12129,13614,13051,13376,12906,13400,13521,13177,13765,12929,13669
2022,13713,12158,13631,13290,13599,13188,13469,13529,13082,13308,13144,13614
2023,13434,12203,13544,13206,13555,13110,13539,13512,13119,13652,13057,13382
2024,13394,12787,13490,12916,13723,13086,13541,13596,13108,13662,13145,13364
2025,9764,8337,9718,1272,1275,1290,1267,1272,1278,1275,1274,1321


In [ ]:
sales.groupby("Year")["UnitsSold"].sum()

,UnitsSold
Year,
2020,134638
2021,178053
2022,178863
2023,178617
2024,178887
2025,44022


In [ ]:
sales_2025 = sales[sales["SaleDate"].dt.year == 2025]

sales_2025.groupby(
    sales_2025["SaleDate"].dt.month
)["DealerCode"].nunique()

,DealerCode
SaleDate,
1,16
2,16
3,16
4,16
5,16
6,16
7,16
8,16
9,16


In [ ]:
sales_2024 = sales[sales["SaleDate"].dt.year == 2024]

sales_2024.groupby(
    sales_2024["SaleDate"].dt.month
)["DealerCode"].nunique()

,DealerCode
SaleDate,
1,16
2,16
3,16
4,16
5,16
6,16
7,16
8,16
9,16


In [ ]:
sales_2025.groupby(
    sales_2025["SaleDate"].dt.month
)["ProductID"].nunique()

,ProductID
SaleDate,
1,57
2,57
3,57
4,57
5,57
6,57
7,57
8,57
9,57


In [ ]:
sales_2025.groupby(
    [sales_2025["SaleDate"].dt.month, "Channel"]
).size().unstack(fill_value=0)

Channel,Arena,NEXA
SaleDate,,
1,4911,4853
2,4184,4153
3,4873,4845
4,630,642
5,621,654
6,669,621
7,614,653
8,640,632
9,645,633


In [ ]:
sales.groupby(
    [sales["SaleDate"].dt.year, "Channel"]
).size().unstack(fill_value=0)

Channel,Arena,NEXA
SaleDate,,
2020,60210,60155
2021,79704,79425
2022,79619,80106
2023,79414,79899
2024,80225,79587
2025,19732,19611


In [ ]:
sales.groupby(
    [sales["SaleDate"].dt.year, "DealerCode"]
).size().unstack(fill_value=0)

DealerCode,AMB,AMR,CMP,DDM,IND,KAT,KHV,KTL,MAG,PRM,PRT,RAN,SAG,TRS,VAR,VPL
SaleDate,,,,,,,,,,,,,,,,
2020,7506,7572,7667,7511,7561,7508,7385,7398,7649,7478,7551,7583,7559,7592,7345,7500
2021,9955,9925,10116,9920,9869,9834,10050,10120,9850,9984,9894,9965,10007,9809,9970,9861
2022,9986,10017,10127,10078,9934,9964,9862,10015,9978,9910,10096,10002,9928,10064,9898,9866
2023,9992,9909,9891,9958,9902,10017,10037,9866,9836,10123,9985,10001,9912,9896,10041,9947
2024,10085,10216,10058,9872,9813,9945,9914,10033,9894,10017,10115,9993,9816,10065,9967,10009
2025,2538,2497,2418,2382,2436,2489,2504,2544,2497,2426,2444,2512,2312,2513,2413,2418


# Final post cleaning validation

In [ ]:
print("Sales shape:", sales.shape)
print("Products shape:", product.shape)
print("Dealers shape:", dealers.shape)
sales["SatisfactionScore"].astype('Int64')
sales["BookingToDeliveryDays"].astype('Int64')

print("\nDuplicate InvoiceIDs:", sales["InvoiceID"].duplicated().sum())
print("Duplicate ProductIDs:", product["ProductID"].duplicated().sum())
print("Duplicate DealerCodes:", dealers["DealerCode"].duplicated().sum())

print("\nMissing ProductIDs:", sales["ProductID"].isna().sum())
print("Missing DealerCodes:", sales["DealerCode"].isna().sum())

print("\nData types:")
print(sales.dtypes)

Sales shape: (797687, 22)
Products shape: (57, 6)
Dealers shape: (16, 4)

Duplicate InvoiceIDs: 0
Duplicate ProductIDs: 0
Duplicate DealerCodes: 0

Missing ProductIDs: 0
Missing DealerCodes: 0

Data types:
SaleDate                 datetime64[ns]
InvoiceID                        object
ProductID                        object
FuelType                         object
Transmission                     object
Color                            object
DealerCode                       object
Channel                          object
UnitsSold                         int64
ExShowroomPrice                   int64
DiscountApplied                   int64
OnRoadPrice                       int64
PaymentMode                      object
FinancePartner                   object
CustomerType                     object
SatisfactionScore               float64
BookingToDeliveryDays           float64
Year                              int32
Month                             int32
SatisfactionMissing               

dropping the satifaction missing and onroad premium

In [ ]:
sales_sql = sales.drop(columns=["SatisfactionMissing", "OnRoad_Premium"])
sales_sql.to_csv("sales_cleaned.csv",index = False)